#*Libraries*

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.3 MB/s eta 0:00:00


In [ ]:
!pip install exifread

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 1.6 MB/s eta 0:00:00


# Imported model

In [ ]:
import torch as pt
import torch
from ultralytics import YOLO
import ultralytics

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
model = torch.load("/content/drive/MyDrive/Colab Notebooks/FYP/best2.pt",weights_only=False)

In [ ]:
model1 = YOLO("/content/drive/MyDrive/Colab Notebooks/FYP/best2.pt")

In [ ]:
a = model1.predict("/content/drive/MyDrive/Colab Notebooks/FYP/images/DJI_0851.JPG" , save=True)


image 1/1 /content/drive/MyDrive/Colab Notebooks/FYP/images/DJI_0851.JPG: 448x640 168 trees, 357.0ms
Speed: 19.2ms preprocess, 357.0ms inference, 62.0ms postprocess per image at shape (1, 3, 448, 640)
Results saved to /content/runs/detect/predict


In [ ]:
import csv
import os
csv_filepath = "/content/drive/MyDrive/Colab Notebooks/FYP/img_metadata.csv"
with open(csv_filepath, mode='w', newline='') as csv_file:
        csv_writer = csv.writer(csv_file)
        # Write header row
        csv_writer.writerow(["latitude","longitude"])
        csv_writer.writerow(["30.052369583333334" ,"74.35073927777778"])

In [ ]:
import csv
import os

# test_images = '/content/drive/MyDrive/drone/Chandrala_60v'
output_dir = '/content/runs/detect/predict'
# os.makedirs(output_dir, exist_ok=True)

# # Load a model
# model = YOLO("/content/drive/MyDrive/Fine_Tuned_Model_YOLO/best2.pt") #path of the output training


# # Set a confidence threshold (e.g., 0.5 for 50%)
# model.conf = 0.3  # Adjust this as needed

# # Load and perform object detection on test images
# results = model(test_images, conf=model.conf, save=True)  # Apply threshold and save results

# Save detections to separate CSV files
for result in a:
    image_name = os.path.basename(result.path)  # Extract image name from the path
    csv_filename = os.path.splitext(image_name)[0] + '.csv'  # Replace extension with .csv
    csv_filepath = os.path.join(output_dir, csv_filename)

    # Get image dimensions
    img_width, img_height = result.orig_shape[1], result.orig_shape[0]  # Width, Height of the image

    with open(csv_filepath, mode='w', newline='') as csv_file:
        csv_writer = csv.writer(csv_file)
        # Write header row
        csv_writer.writerow(["Class_id", "X_center", "Y_center", "Width", "Height"])

        # Write detections for this image
        boxes = result.boxes  # Detected boxes
        for box in boxes:
            cls = box.cls.item()  # Class index
            xmin, ymin, xmax, ymax = box.xyxy[0].tolist()  # Bounding box coordinates

            # Convert to YOLO input format (normalized center-based coordinates)
            width = xmax - xmin
            height = ymax - ymin
            x_center = xmin + width / 2
            y_center = ymin + height / 2

            # Normalize the coordinates
            x_center_norm = x_center / img_width
            y_center_norm = y_center / img_height
            width_norm = width / img_width
            height_norm = height / img_height

            # Write normalized detection to CSV
            csv_writer.writerow([cls, x_center_norm, y_center_norm, width_norm, height_norm])

    print(f"Detections for {image_name} saved to {csv_filepath}")


Detections for DJI_0851.JPG saved to /content/runs/detect/predict/DJI_0851.csv


# The calculating pixel size (Ground Sampling Distance (GSD))

In [ ]:
#The calculating pixel size (Ground Sampling Distance (GSD)) is to determine the real world spatial resolution of an drone imagery.

#Imported Library
import math
import csv
import exifread
from pyproj import Transformer

# Initialize Transformer for UTM conversion
transformer_wgs84_to_utm = Transformer.from_crs("epsg:4326", "epsg:32644", always_xy=True)  # Change UTM zone as needed
transformer_utm_to_wgs84 = Transformer.from_crs("epsg:32644", "epsg:4326", always_xy=True)

def wgs84_to_utm(lat, lon): #Convert WGS-84 latitude/longitude to UTM easting and northing.
    easting, northing = transformer_wgs84_to_utm.transform(lon, lat)
    return easting, northing

def utm_to_wgs84(easting, northing):#Convert UTM easting/northing to WGS-84 latitude/longitude.(optional for large area)
    lon, lat = transformer_utm_to_wgs84.transform(easting, northing)
    return lat, lon

def extract_gps_and_camera_metadata(image_path):#Extract metadata including GPS and camera parameters.

    with open(image_path, 'rb') as f:
        tags = exifread.process_file(f)

    def dms_to_dd(dms, ref):
        degrees = dms.values[0].num / dms.values[0].den
        minutes = dms.values[1].num / dms.values[1].den
        seconds = dms.values[2].num / dms.values[2].den
        dd = degrees + (minutes / 60) + (seconds / 3600)
        return dd if ref in ['N', 'E'] else -dd

    metadata = {
        "latitude": None,
        "longitude": None,
        "altitude": None,
        "image_width": None,
        "image_height": None,
        "focal_length_mm": None,
    }

    try:
        if 'GPS GPSLatitude' in tags and 'GPS GPSLatitudeRef' in tags:
            metadata["latitude"] = dms_to_dd(tags['GPS GPSLatitude'], tags['GPS GPSLatitudeRef'].values)
        if 'GPS GPSLongitude' in tags and 'GPS GPSLongitudeRef' in tags:
            metadata["longitude"] = dms_to_dd(tags['GPS GPSLongitude'], tags['GPS GPSLongitudeRef'].values)
        if 'GPS GPSAltitude' in tags:
            metadata["altitude"] = tags['GPS GPSAltitude'].values[0].num / tags['GPS GPSAltitude'].values[0].den
        metadata["image_width"] = tags['EXIF PixelXDimension'].values[0] if 'EXIF PixelXDimension' in tags else 5472
        metadata["image_height"] = tags['EXIF PixelYDimension'].values[0] if 'EXIF PixelYDimension' in tags else 3648
        if 'EXIF FocalLength' in tags:
            metadata["focal_length_mm"] = tags['EXIF FocalLength'].values[0].num / tags['EXIF FocalLength'].values[0].den

        # Convert latitude and longitude to UTM
        if metadata["latitude"] and metadata["longitude"]:
            metadata["utm_easting"], metadata["utm_northing"] = wgs84_to_utm(metadata["latitude"], metadata["longitude"])

    except Exception as e:
        print(f"Warning: Failed to extract metadata: {e}")
    return metadata

def calculate_gsd(metadata, sensor_width_mm, geoid_height):#https://view.officeapps.live.com/op/view.aspx?src=https%3A%2F%2Fdata.pix4d.com%2Fmisc%2FKB%2Fdocuments%2FPix4D_GSD_Calculator.xlsx&wdOrigin=BROWSELINK
    """
    Calculate GSD using the standard formula.
    GSD = (Altitude * Sensor Width) / (Focal Length * Image Width)
    """
    try:
        # Extract relevant metadata
        altitude_utm = metadata["altitude"] - geoid_height
        print(f"Converted UTM Altitude (MSL): {altitude_utm} meters")

        focal_length_mm = metadata["focal_length_mm"]
        image_width = metadata["image_width"]

        # Apply the standard GSD formula
        gsd = (altitude_utm * sensor_width_mm) / (focal_length_mm * image_width)

        # Convert GSD to degrees (approximation at the equator)
        meters_per_degree = 111320  # Average meters per degree at the equator
        gsd_degrees = gsd / meters_per_degree

        return gsd, gsd_degrees

    except Exception as e:
        print(f"Error in calculating GSD: {e}")
        return None, None

# Example usage
if __name__ == "__main__":
    image_path = r"/content/drive/MyDrive/Colab Notebooks/FYP/images/DJI_0851.JPG"
    sensor_width_mm = 13.2
    geoid_height = 105.8 #changes according to state

    metadata = extract_gps_and_camera_metadata(image_path)

    if metadata:
        print("Extracted Metadata:")
        for key, value in metadata.items():
            print(f"{key}: {value}")#Metadata

        gsd_meters, gsd_degrees = calculate_gsd(metadata, sensor_width_mm, geoid_height)

        if gsd_meters and gsd_degrees:
            print(f"Calculated GSD: {gsd_meters} meters/pixel")
            print(f"Calculated GSD: {gsd_degrees} degrees/pixel")

Extracted Metadata:
latitude: 30.052369583333334
longitude: 74.35073927777778
altitude: 251.0
image_width: 5472
image_height: 3648
focal_length_mm: 10.26
utm_easting: -141690.83807745017
utm_northing: 3343288.132679817
Converted UTM Altitude (MSL): 145.2 meters
Calculated GSD: 0.034138709346465575 meters/pixel
Calculated GSD: 3.0667184105700303e-07 degrees/pixel


# The Calculate the Yaw Angle Between Two Adjacent Images

In [ ]:
#The Calculate the Yaw Angle Between Two Adjacent Images
#Imported Library
import os
import exifread
import math

# Extract GPS Coordinates from an Image
def extract_gps_coordinates(image_path):
    with open(image_path, 'rb') as f:
        tags = exifread.process_file(f)

    def dms_to_dd(dms, ref):
        degrees = dms.values[0].num / dms.values[0].den
        minutes = dms.values[1].num / dms.values[1].den
        seconds = dms.values[2].num / dms.values[2].den
        dd = degrees + (minutes/60) + (seconds/3600)
        return dd if ref in ['N', 'E'] else -dd

    latitude, longitude = None, None
    try:
        if 'GPS GPSLatitude' in tags and 'GPS GPSLatitudeRef' in tags:
            latitude = dms_to_dd(tags['GPS GPSLatitude'], tags['GPS GPSLatitudeRef'].values)
        if 'GPS GPSLongitude' in tags and 'GPS GPSLongitudeRef' in tags:
            longitude = dms_to_dd(tags['GPS GPSLongitude'], tags['GPS GPSLongitudeRef'].values)
    except Exception as e:
        print(f'Error extracting GPS from {image_path} {e}')

    return latitude, longitude

# Calculate Yaw Angle Between Two Images
def calculate_yaw_angle(lat1, lon1, lat2, lon2):
    delta_lon = lon2 - lon1
    delta_lat = lat2 - lat1
    #print('delta_lon=',delta_lon, 'delta_lat=',delta_lat)
    yaw_radians = math.atan2(delta_lon, delta_lat)
    yaw_degrees = math.degrees(yaw_radians)
    yaw_degrees = (yaw_degrees + 360) % 360  # Normalize to [0, 360]
    #print('yaw degrees =',yaw_degrees)
    return yaw_degrees

# Batch processing
def process_folder(folder_path): # List all image files in the folder
    image_files = sorted([
        os.path.join(folder_path, f) for f in os.listdir(folder_path)
        if f.lower().endswith(('.jpg', '.jpeg'))
    ])
    if len(image_files) < 2:
        print('Error At least two images are required to calculate yaw angles.')
        return

    # Extract GPS for each image and calculate yaw angle between Adjacent images
    for i in range(len(image_files) - 1):
        lat1, lon1 = extract_gps_coordinates(image_files[i])
        lat2, lon2 = extract_gps_coordinates(image_files[i + 1])
        if None in (lat1, lon1, lat2, lon2):
            print(f'Skipping image pair {image_files[i]} and {image_files[i+1]} due to missing GPS data.')
            continue
        yaw_angle = calculate_yaw_angle(lat1, lon1, lat2, lon2)
        print(f'Yaw Angle between {os.path.basename(image_files[i])} and {os.path.basename(image_files[i + 1])} {yaw_angle} degrees')

# Source File
folder_path = r"/content/drive/MyDrive/Colab Notebooks/FYP/images" #Folder Path
process_folder(folder_path)

Yaw Angle between DJI_0851.JPG and DJI_0852.JPG 349.59412695578686 degrees
Yaw Angle between DJI_0852.JPG and DJI_0853.JPG 349.6550745863577 degrees
Yaw Angle between DJI_0853.JPG and DJI_0854.JPG 349.6842286034888 degrees
Yaw Angle between DJI_0854.JPG and DJI_0855.JPG 349.6479184472093 degrees


# Draft Final Geolocation process based

In [ ]:
'''
Draft Final Geolocation process based "Article (Deep-Learning-Based Automated Palm Tree Counting and
Geolocation in Large Farms from Aerial Geotagged Images)"
'''
#Imported Necessary Libraries
import math
import csv
import exifread

# Function to extract metadata
def extract_gps_and_camera_metadata(image_path):
    """
    Extract metadata including GPS and camera parameters.
    """
    with open(image_path, 'rb') as f:
        tags = exifread.process_file(f)

    def dms_to_dd(dms, ref):
        degrees = dms.values[0].num / dms.values[0].den
        minutes = dms.values[1].num / dms.values[1].den
        seconds = dms.values[2].num / dms.values[2].den
        dd = degrees + (minutes / 60) + (seconds / 3600)
        return dd if ref in ['N', 'E'] else -dd

    metadata = {
        "latitude": None,
        "longitude": None,
        "altitude": None,
        "image_width": None,
        "image_height": None,
        "focal_length": None,
    }

    try:
        if 'GPS GPSLatitude' in tags and 'GPS GPSLatitudeRef' in tags:
            metadata["latitude"] = dms_to_dd(tags['GPS GPSLatitude'], tags['GPS GPSLatitudeRef'].values)
        if 'GPS GPSLongitude' in tags and 'GPS GPSLongitudeRef' in tags:
            metadata["longitude"] = dms_to_dd(tags['GPS GPSLongitude'], tags['GPS GPSLongitudeRef'].values)
        if 'GPS GPSAltitude' in tags:
            metadata["altitude"] = tags['GPS GPSAltitude'].values[0].num / tags['GPS GPSAltitude'].values[0].den
        metadata["image_width"] = tags['EXIF PixelXDimension'].values[0] if 'EXIF PixelXDimension' in tags else 5472 #Check Dimension of Image
        metadata["image_height"] = tags['EXIF PixelYDimension'].values[0] if 'EXIF PixelYDimension' in tags else 3648 #Check Dimension of Image
    except Exception as e:
        print(f"Warning: Failed to extract metadata: {e}")
    return metadata

# Denormalize YOLO coordinates
def denormalize_coordinates(x_norm, y_norm, Iw, Ih):
    """
    Convert normalized coordinates to pixel coordinates.
    """
    x_pixel = x_norm * Iw
    y_pixel = y_norm * Ih
    return x_pixel, y_pixel

# 2.5.1. Calculation of the Distance to Image Center
def calculate_distances_to_center(x, y, xc, yc, pixel_size):
    """
    Compute real-world distances from the center.
    """
    dx = (x - xc) * pixel_size
    dy = (yc - y) * pixel_size
    return dx, dy

# Apply yaw rotation
def apply_yaw_rotation(dx, dy, yaw_angle):
    """
    Rotate distances based on the yaw angle.
    """
    yaw_radians = math.radians(yaw_angle)
    Dx = dx * math.cos(yaw_radians) - dy * math.sin(yaw_radians)
    Dy = dx * math.sin(yaw_radians) + dy * math.cos(yaw_radians)
    return Dx, Dy

# 2.5.2. Distance Correction
def apply_distance_correction(dx, dy, h, H):
    """
    Apply distance correction based on object height.
    """
    # Calculate total distance D
    D = math.sqrt(dx**2 + dy**2)

    # Check if height is available, otherwise skip correction
    if h > 0 and H > 0:
        d = (h / H) * D
    else:
        d = 0  # No correction applied if h or H is missing or invalid

    # Apply correction
    if D > 0:  # Avoid division by zero
        Dc_x = dx * (1 - d / D)
        Dc_y = dy * (1 - d / D)
    else:
        Dc_x, Dc_y = dx, dy  # No correction needed if D = 0

    return Dc_x, Dc_y

# 2.5.3. Conversion to GPS Coordinates(WGS-84)
def convert_to_geographic_coordinates(lat_center, lon_center, Dx, Dy):
    """
    Transform corrected distances to geographic coordinates.
    """
    new_lat = lat_center + Dy
    new_lon = lon_center + Dx / math.cos(math.radians(lat_center))
    return new_lat, new_lon

# Process YOLO-detected objects with per-object height
def process_detected_points_with_correction(metadata, pixel_size, yaw_angle, prediction_file, output_csv, object_heights=None):
    """
    Process detected objects, apply distance correction, and calculate geographic coordinates.
    """
    lat_center = metadata["latitude"] #23.3438481 #
    lon_center = metadata["longitude"] #72.7883352  #
    H = metadata["altitude"]
    Iw = metadata["image_width"]
    Ih = metadata["image_height"]

    results = []
    with open(prediction_file, 'r') as f:
        lines = f.readlines()
    z=0
    for idx, line in enumerate(lines):
        if z==0:
          z+=1
          continue
        # Parse YOLO format: <class_id> <x_norm> <y_norm> <width> <height>
        parts = line.strip().split(sep= ',')
        class_id, x_norm, y_norm, width, height = map(float, parts)

        # Denormalize coordinates
        x_pixel, y_pixel = denormalize_coordinates(x_norm, y_norm, Iw, Ih)

        # Calculate distances to the center
        xc, yc = Iw / 2, Ih / 2
        dx, dy = calculate_distances_to_center(x_pixel, y_pixel, xc, yc, pixel_size)

        # Apply yaw rotation
        Dx, Dy = apply_yaw_rotation(dx, dy, yaw_angle)

        # Use object-specific height if available
        h = object_heights[idx] if object_heights and idx < len(object_heights) else 0  # Default to 0 if height not provided

        # Apply distance correction
        Dc_x, Dc_y = apply_distance_correction(Dx, Dy, h, H)

        # Convert to geographic coordinates
        new_lat, new_lon = convert_to_geographic_coordinates(lat_center, lon_center, Dc_x, Dc_y)

        # Append result
        results.append({
            "class_id": int(class_id),
            "latitude": new_lat,
            "longitude": new_lon
        })

    # Save results to CSV
    with open(output_csv, mode='w', newline='') as csvfile:
        fieldnames = ['class_id', 'latitude', 'longitude']#Can add class_nameas per requirment
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    print(f"Georeferenced detected points with correction saved to: {output_csv}")

# Main Workflow
if __name__ == "__main__":
    image_path = r"/content/drive/MyDrive/Colab Notebooks/FYP/images/DJI_0851.JPG"
    prediction_file = r"/content/runs/detect/predict/DJI_0851.csv"
    output_csv = r"/content/drive/MyDrive/Colab Notebooks/FYP/output?/o1.csv"
    yaw_angle = -286.9849298089436 #-289.6752727076506 #-289.6752727076506#  # Yaw angle in degrees
    pixel_size =1.4920605209950548e-07 #1.486888734093859e-07 # #1.4894194390013116E-07 # Pixel size in degrees

    # Example object heights (replace with actual heights or set to None)
    object_heights = [10.0]  # Replace with the correct number of objects and heights

    # Extract metadata
    metadata = extract_gps_and_camera_metadata(image_path)

    # Process detected points with correction
    process_detected_points_with_correction(metadata, pixel_size, yaw_angle, prediction_file, output_csv, object_heights)

Georeferenced detected points with correction saved to: /content/drive/MyDrive/Colab Notebooks/FYP/output?/o1.csv


# duplicates

In [ ]:
import csv
import os
import math
from typing import List, Dict, Tuple

def deg2rad(deg: float) -> float:
    """Convert degrees to radians"""
    return deg * (math.pi / 180.0)

def local_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Calculate the local distance between two points using their latitude and longitude.
    Returns distance in meters.
    """
    # Convert degrees to radians
    lat1, lon1 = deg2rad(lat1), deg2rad(lon1)
    lat2, lon2 = deg2rad(lat2), deg2rad(lon2)

    # Earth's radius in meters
    R = 6371000

    # Compute differences
    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1
    avg_lat = (lat1 + lat2) / 2

    # Convert to meters
    dx = R * math.cos(avg_lat) * delta_lon  # Longitude difference in meters
    dy = R * delta_lat  # Latitude difference in meters

    # Compute Euclidean distance
    return math.sqrt(dx**2 + dy**2)

def read_csv_file(filename: str) -> List[Dict]:
    """Read CSV file and return list of dictionaries"""
    data = []
    with open(filename, 'r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            data.append({
                'class_id': row['class_id'],
                'latitude': float(row['latitude']),
                'longitude': float(row['longitude'])
            })
    return data

def write_csv_file(filename: str, data: List[Dict]) -> None:
    """Write data to CSV file"""
    fieldnames = ['class_id', 'latitude', 'longitude']
    with open(filename, 'w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)

def is_point_duplicate(point: Dict, existing_points: List[Dict], threshold: float = 5.0) -> bool:
    """
    Check if a point is duplicate or too close to any existing point
    Returns True if duplicate or too close
    """
    for existing in existing_points:
        distance = local_distance(
            point['latitude'], point['longitude'],
            existing['latitude'], existing['longitude']
        )
        if distance < threshold:
            return True
    return False

def process_csv_files(csv_folder: str, threshold: float = 5.0) -> None:
    """
    Process all CSV files in the folder and create new files without duplicates,
    comparing only with the previous file.
    """
    # Get all CSV files in the folder
    csv_files = sorted([f for f in os.listdir(csv_folder) if f.endswith('.csv')])

    if not csv_files:
        print("No CSV files found in the folder")
        return

    # Process files sequentially
    for i in range(len(csv_files)):
        current_file = os.path.join(csv_folder, csv_files[i])
        current_data = read_csv_file(current_file)

        # Create new filename for processed data
        new_filename = os.path.join(csv_folder, f"T3m_{csv_files[i]}")

        # If this is the first file, just save it as is
        if i == 0:
            write_csv_file(new_filename, current_data)
            continue

        # Read all previous processed files
        all_previous_points = []
        for j in range(i):
            prev_processed_file = os.path.join(csv_folder, f"T3m_{csv_files[j]}")
            all_previous_points.extend(read_csv_file(prev_processed_file))

        # Filter out duplicates and close points
        filtered_data = []
        for point in current_data:
            if point['class_id'] == '1' or point['class_id'] == 1:
                continue
            if not is_point_duplicate(point, all_previous_points, threshold):
                filtered_data.append(point)


        # Save filtered data
        write_csv_file(new_filename, filtered_data)

        print(f"Processed {csv_files[i]}: {len(current_data)} points -> {len(filtered_data)} points")

# Example usage
if __name__ == "__main__":
    csv_folder = r"/content/drive/MyDrive/Colab Notebooks/FYP/output?"
    threshold = 3.0  # Distance threshold in meters
    process_csv_files(csv_folder, threshold)

# final tree count

In [ ]:
import csv
import os
import math
from typing import List, Dict, Tuple

def deg2rad(deg: float) -> float:
    """Convert degrees to radians"""
    return deg * (math.pi / 180.0)

def local_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Calculate the local distance between two points using their latitude and longitude.
    Returns distance in meters.
    """
    # Convert degrees to radians
    lat1, lon1 = deg2rad(lat1), deg2rad(lon1)
    lat2, lon2 = deg2rad(lat2), deg2rad(lon2)

    # Earth's radius in meters
    R = 6371000

    # Compute differences
    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1
    avg_lat = (lat1 + lat2) / 2

    # Convert to meters
    dx = R * math.cos(avg_lat) * delta_lon  # Longitude difference in meters
    dy = R * delta_lat  # Latitude difference in meters

    # Compute Euclidean distance
    return math.sqrt(dx**2 + dy**2)

def read_csv_file(filename: str) -> List[Dict]:
    """Read CSV file and return list of dictionaries"""
    data = []
    with open(filename, 'r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            data.append({
                'class_id': row['class_id'],
                'latitude': float(row['latitude']),
                'longitude': float(row['longitude'])
            })
    return data

def write_csv_file(filename: str, data: List[Dict]) -> None:
    """Write data to CSV file"""
    fieldnames = ['class_id', 'latitude', 'longitude']
    with open(filename, 'w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)

def is_point_duplicate(point: Dict, existing_points: List[Dict], threshold: float = 5.0) -> bool:
    """
    Check if a point is duplicate or too close to any existing point
    Returns True if duplicate or too close
    """
    for existing in existing_points:
        distance = local_distance(
            point['latitude'], point['longitude'],
            existing['latitude'], existing['longitude']
        )
        if distance < threshold:
            return True
    return False

def process_csv_files(csv_folder: str, threshold: float = 5.0) -> Dict[str, int]:
    """
    Process all CSV files in the folder and create new files without duplicates,
    comparing only with the previous file.
    Returns a dictionary with point counts for original and final files.
    """
    # Get all CSV files in the folder
    csv_files = sorted([f for f in os.listdir(csv_folder) if f.endswith('.csv')])

    if not csv_files:
        print("No CSV files found in the folder")
        return {"original_total": 0, "final_total": 0}

    original_total = 0
    final_total = 0

    # Process files sequentially
    for i in range(len(csv_files)):
        current_file = os.path.join(csv_folder, csv_files[i])
        current_data = read_csv_file(current_file)
        original_total += len(current_data)

        # Create new filename for processed data
        new_filename = os.path.join(csv_folder, f"Final_{csv_files[i]}")

        # If this is the first file, just save it as is
        if i == 0:
            write_csv_file(new_filename, current_data)
            final_total += len(current_data)
            continue

        # Read all previous processed files
        all_previous_points = []
        for j in range(i):
            prev_processed_file = os.path.join(csv_folder, f"T3m_{csv_files[j]}")
            all_previous_points.extend(read_csv_file(prev_processed_file))

        # Filter out duplicates and close points
        filtered_data = []
        for point in current_data:
            if point['class_id'] == '1' or point['class_id'] == 1:
                continue
            if not is_point_duplicate(point, all_previous_points, threshold):
                filtered_data.append(point)

        # Save filtered data
        write_csv_file(new_filename, filtered_data)
        final_total += len(filtered_data)

        print(f"Processed {csv_files[i]}: {len(current_data)} points -> {len(filtered_data)} points")

    print(f"\nSummary:")
    print(f"Total points in original files: {original_total}")
    print(f"Total points in final files: {final_total}")
    print(f"Points removed: {original_total - final_total}")

    return {
        "original_total": original_total,
        "final_total": final_total,
        "points_removed": original_total - final_total
    }

# Example usage
if __name__ == "__main__":
    csv_folder = r"/content/drive/MyDrive/Colab Notebooks/FYP/output?"  # Replace with your folder path
    threshold = 5.0  # Distance threshold in meters
    results = process_csv_files(csv_folder, threshold)


Summary:
Total points in original files: 168
Total points in final files: 168
Points removed: 0
